<a href="https://colab.research.google.com/github/Podrimate/THz_sim_application/blob/main/notebooks/THzSim_User_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# THzSim User Workflow

Code-cell-driven THz simulation and study workflow for arbitrary isotropic multilayer samples.

## 1. Install And Import

Run the next cell first. In Colab it installs the package from GitHub and prints the active version numbers.

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
PACKAGE_IMPORT_OK = False
IMPORT_EXCEPTION = None
DEFAULT_PIP_SPEC = 'https://github.com/Podrimate/THz_sim_application/archive/refs/heads/main.zip'

def try_import_runtime_dependencies():
    global IMPORT_EXCEPTION
    try:
        import numpy  # noqa: F401
        import scipy  # noqa: F401
        import matplotlib  # noqa: F401
        return True
    except Exception as exc:
        IMPORT_EXCEPTION = exc
        return False

def try_import_thzsim2():
    global IMPORT_EXCEPTION
    try:
        importlib.invalidate_caches()
        import thzsim2  # noqa: F401
        return thzsim2
    except Exception as exc:
        IMPORT_EXCEPTION = exc
        return None

if IN_COLAB:
    pip_spec = os.environ.get('THZSIM_PIP_SPEC', DEFAULT_PIP_SPEC).strip()
    subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'thzsim2'])
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '--upgrade',
        '--force-reinstall',
        '--no-cache-dir',
        '--no-deps',
        pip_spec,
    ])
    sys.modules.pop('thzsim2', None)
    sys.modules.pop('thzsim2.notebook_api', None)

if not try_import_runtime_dependencies():
    raise RuntimeError(
        'The Python runtime dependencies are currently broken. '
        f'Last import error: {IMPORT_EXCEPTION!r}. '
        'In Colab use Runtime -> Factory reset runtime, then rerun the first cell. '
        "The notebook now installs only thzsim2 and leaves Colab's NumPy/SciPy/Matplotlib in place."
    )

thzsim2 = try_import_thzsim2()
PACKAGE_IMPORT_OK = thzsim2 is not None

if not PACKAGE_IMPORT_OK:
    for candidate in [Path.cwd(), Path.cwd().parent]:
        if (candidate / 'thzsim2').exists():
            sys.path.insert(0, str(candidate))
            thzsim2 = try_import_thzsim2()
            if thzsim2 is not None:
                PACKAGE_IMPORT_OK = True
                break

if not PACKAGE_IMPORT_OK:
    raise RuntimeError(
        'Could not import thzsim2. '
        f'Last import error: {IMPORT_EXCEPTION!r}. '
        'In Colab use Runtime -> Factory reset runtime and rerun the first cell; '
        'locally open the notebook inside the repo.'
    )

import matplotlib
import numpy
import scipy

print(f'Using thzsim2 from: {thzsim2.__file__}')
print(f'thzsim2 version: {getattr(thzsim2, "__version__", "unknown")}')
print(f'numpy version: {numpy.__version__}')
print(f'scipy version: {scipy.__version__}')
print(f'matplotlib version: {matplotlib.__version__}')

In [ ]:
from copy import deepcopy
from pathlib import Path
from pprint import pprint
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from thzsim2.io.trace_csv import read_trace_csv, write_trace_csv
from thzsim2.notebook_api import (
    Measurement,
    create_run_output_dir,
    ensure_builtin_material_file,
    estimate_study_runtime,
    find_nearest_study_row,
    fit_param,
    generate_reference_pulse,
    inspect_trace_input,
    layers_from_definition,
    plot_study_case_detail,
    plot_study_heatmap_selector,
    plot_trace_pair_preview,
    plot_trace_preview,
    prepare_reference,
    prepare_trace_pair_for_fit,
    preview_sample_response,
    preview_study_noise,
    run_measured_fit,
    run_study_with_progress,
    save_fit_setup_snapshot,
    save_study_setup_snapshot,
    sweep_axis,
    trace_spectrum,
    write_python_snapshot,
)

IN_COLAB = 'google.colab' in sys.modules

def find_repo_path(relative_path):
    for candidate in [Path.cwd(), Path.cwd().parent]:
        candidate_path = candidate / relative_path
        if candidate_path.exists():
            return candidate_path.resolve()
    return None

def upload_single_csv(target_dir, *, prompt):
    print(prompt)
    try:
        from google.colab import files
    except Exception as exc:
        raise RuntimeError('Colab upload is only available inside Google Colab.') from exc
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file was uploaded.')
    name, payload = next(iter(uploaded.items()))
    path = Path(target_dir) / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(payload)
    return path.resolve()

def print_mapping(title, mapping):
    display(Markdown(f'**{title}**'))
    for key, value in mapping.items():
        print(f'- {key}: {value}')

workflow_root = Path.cwd() / 'thz_sim_workflow_outputs'
workflow_root.mkdir(parents=True, exist_ok=True)
LOCAL_TEST_DATA = find_repo_path('Test_data_for_fitter')

## 2. Run Setup, Choose Or Generate The Reference Trace, And Preview The Data

Edit the next cell and rerun until the plots look right.

Design philosophy for this section:
- always inspect the trace before moving on
- if you upload, look at both the raw and prepared trace
- if you generate, treat the generated trace as data too and inspect it the same way
- use the FFT preview whenever you want to check bandwidth, ringing, or phase behavior

In [ ]:
run_name = 'study_run'
measurement_mode = 'transmission'  # 'transmission' or 'reflection'

reference_source = 'generate'  # 'generate' or 'upload'
use_colab_upload_for_reference = IN_COLAB and reference_source == 'upload'
reference_csv_path = ''
reference_time_column = None
reference_signal_column = None

show_reference_fft = True
reference_fft_limits_thz = None  # example: (0.0, 4.0)

generated_trace = {
    'model': 'sech_carrier',
    'sample_count': 4096,
    'dt_ps': 0.05,
    'time_center_ps': 20.0,
    'pulse_center_ps': 20.0,
    'tau_ps': 0.35,
    'f0_thz': 0.9,
    'amp': 1.0,
    'phi_rad': 0.0,
    'pad_factor': 1,
}

In [ ]:
run_dir = create_run_output_dir(run_name, root=workflow_root)
uploads_dir = run_dir / 'uploads'
python_dir = run_dir / 'python_snapshots'
config_dir = run_dir / 'configs'
materials_dir = run_dir / 'materials'
for directory in (uploads_dir, python_dir, config_dir, materials_dir):
    directory.mkdir(parents=True, exist_ok=True)

if reference_source == 'upload':
    selected_reference_csv = Path(reference_csv_path).expanduser().resolve() if reference_csv_path else None
    if use_colab_upload_for_reference:
        selected_reference_csv = upload_single_csv(uploads_dir, prompt='Upload the measured reference CSV.')
    if selected_reference_csv is None:
        raise RuntimeError('Please provide a reference CSV path or upload a file.')
    reference_info = inspect_trace_input(
        selected_reference_csv,
        time_column=reference_time_column,
        signal_column=reference_signal_column,
    )
    reference_trace_input = reference_info['prepared_trace']
    reference_setup_snapshot = {
        'kind': 'measured_csv',
        'path': str(selected_reference_csv),
        'time_column': reference_time_column,
        'signal_column': reference_signal_column,
        'prepare': {
            'output_root': str(run_dir / 'reference_runs'),
            'run_label': run_name,
        },
    }
    print(f'Run directory: {run_dir}')
    print(f'Reference CSV: {selected_reference_csv}')
    plot_trace_preview(reference_info, title_prefix='Reference', show_fft=show_reference_fft, freq_limits_thz=reference_fft_limits_thz)
    print_mapping('Reference Summary', reference_info['summary'])
else:
    reference_trace_input = generate_reference_pulse(**generated_trace)
    reference_info = inspect_trace_input(reference_trace_input)
    generated_reference_csv = run_dir / 'generated_reference_trace.csv'
    write_trace_csv(generated_reference_csv, reference_trace_input)
    reference_setup_snapshot = {
        'kind': 'generated_pulse',
        'generate': deepcopy(generated_trace),
        'prepare': {
            'output_root': str(run_dir / 'reference_runs'),
            'run_label': run_name,
        },
    }
    print(f'Run directory: {run_dir}')
    print(f'Generated reference CSV: {generated_reference_csv}')
    plot_trace_preview(reference_info, title_prefix='Generated Reference', show_fft=show_reference_fft, freq_limits_thz=reference_fft_limits_thz)
    print_mapping('Generated Reference Summary', reference_info['summary'])

write_python_snapshot(
    python_dir / 'part2_reference_setup.py',
    run_name=run_name,
    measurement_mode=measurement_mode,
    reference_source=reference_source,
    reference_csv_path=str(reference_csv_path),
    generated_trace=generated_trace,
    show_reference_fft=show_reference_fft,
    reference_fft_limits_thz=reference_fft_limits_thz,
)

## 3. Sample (Stack) Definition And Preview

Edit the next cell to build the sample. This is the main user-editable stack cell.

How to think about it:
- the list order is the physical propagation order
- `fit_param(...)` marks a parameter that the later study is allowed to recover
- any parameter you leave as a plain number stays fixed
- after the build step, the notebook simulates the sample from the reference and plots the result next to the reference in both time and frequency domains

In [ ]:
si_nk_csv = ensure_builtin_material_file('si_thz_nk.csv', out_dir=materials_dir)
sio2_nk_csv = ensure_builtin_material_file('sio2_lossy_thz_nk.csv', out_dir=materials_dir)

sample_template_name = 'one_layer_drude'

sample_templates = {
    'one_layer_drude': [
        {
            'name': 'drude_film',
            'thickness_um': fit_param(550.0, 250.0, 850.0, label='film_thickness_um'),
            'material': {
                'kind': 'Drude',
                'eps_inf': fit_param(11.0, 4.0, 20.0, label='film_eps_inf'),
                'plasma_freq_thz': fit_param(1.1, 0.01, 5.0, label='film_plasma_freq_thz'),
                'gamma_thz': fit_param(0.08, 0.001, 2.0, label='film_gamma_thz'),
            },
        }
    ],
    'two_layer_lorentz_plus_si': [
        {
            'name': 'lorentz_coating',
            'thickness_um': fit_param(180.0, 60.0, 420.0, label='coating_thickness_um'),
            'material': {
                'kind': 'DrudeLorentz',
                'eps_inf': fit_param(3.4, 1.0, 8.0, label='coating_eps_inf'),
                'drude_plasma_freq_thz': 0.0,
                'drude_gamma_thz': 1.0,
                'oscillators': [
                    {
                        'delta_eps': fit_param(1.2, 0.1, 4.0, label='osc1_delta_eps'),
                        'resonance_thz': fit_param(0.8, 0.2, 1.6, label='osc1_resonance_thz'),
                        'gamma_thz': fit_param(0.12, 0.01, 0.6, label='osc1_gamma_thz'),
                    },
                    {
                        'delta_eps': fit_param(0.8, 0.1, 4.0, label='osc2_delta_eps'),
                        'resonance_thz': fit_param(1.7, 0.6, 2.8, label='osc2_resonance_thz'),
                        'gamma_thz': fit_param(0.2, 0.01, 0.8, label='osc2_gamma_thz'),
                    },
                ],
            },
        },
        {
            'name': 'si_layer',
            'thickness_um': fit_param(500.0, 200.0, 1000.0, label='si_thickness_um'),
            'material': {'kind': 'NKFile', 'path': str(si_nk_csv)},
        },
    ],
    'three_layer_lorentz_plus_si_plus_sio2_backside': [
        {
            'name': 'lorentz_coating',
            'thickness_um': fit_param(180.0, 60.0, 420.0, label='coating_thickness_um'),
            'material': {
                'kind': 'DrudeLorentz',
                'eps_inf': fit_param(3.4, 1.0, 8.0, label='coating_eps_inf'),
                'drude_plasma_freq_thz': 0.0,
                'drude_gamma_thz': 1.0,
                'oscillators': [
                    {
                        'delta_eps': fit_param(1.2, 0.1, 4.0, label='osc1_delta_eps'),
                        'resonance_thz': fit_param(0.8, 0.2, 1.6, label='osc1_resonance_thz'),
                        'gamma_thz': fit_param(0.12, 0.01, 0.6, label='osc1_gamma_thz'),
                    },
                    {
                        'delta_eps': fit_param(0.8, 0.1, 4.0, label='osc2_delta_eps'),
                        'resonance_thz': fit_param(1.7, 0.6, 2.8, label='osc2_resonance_thz'),
                        'gamma_thz': fit_param(0.2, 0.01, 0.8, label='osc2_gamma_thz'),
                    },
                ],
            },
        },
        {
            'name': 'si_layer',
            'thickness_um': fit_param(500.0, 200.0, 1000.0, label='si_thickness_um'),
            'material': {'kind': 'NKFile', 'path': str(si_nk_csv)},
        },
        {
            'name': 'sio2_backside',
            'thickness_um': 1.0e7,
            'material': {'kind': 'NKFile', 'path': str(sio2_nk_csv)},
        },
    ],
}

sample_definition = deepcopy(sample_templates[sample_template_name])
measurement = Measurement(
    mode=measurement_mode,
    angle_deg=0.0,
    polarization='s',
    polarization_mix=None,
    reference_standard={'kind': 'identity'},
)

sample_n_in = 1.0
sample_n_out = 1.0
overlay_imported_nk = True
sample_preview_internal_reflections = 4
show_sample_fft = True
sample_fft_limits_thz = None

reference_result = prepare_reference(reference_trace_input, output_root=run_dir / 'reference_runs', run_label=run_name)
layers = layers_from_definition(sample_definition)
write_python_snapshot(
    python_dir / 'part3_sample_definition.py',
    sample_template_name=sample_template_name,
    sample_definition=sample_definition,
    sample_n_in=sample_n_in,
    sample_n_out=sample_n_out,
    measurement=measurement,
    sample_preview_internal_reflections=sample_preview_internal_reflections,
)

sample_result, sample_preview_simulation, _, _ = preview_sample_response(
    reference_result=reference_result,
    layers=layers,
    out_dir=run_dir / 'sample_preview',
    measurement=measurement,
    n_in=sample_n_in,
    n_out=sample_n_out,
    overlay_imported=overlay_imported_nk,
    max_internal_reflections=sample_preview_internal_reflections,
    show_fft=show_sample_fft,
    freq_limits_thz=sample_fft_limits_thz,
)
print(f'Reference outputs: {reference_result.run_dir}')
print(f'Sample preview outputs: {sample_result.out_dir}')

## 4. Study Setup, Noise Preview, And Runtime Estimate

Edit the next cell until the planned study matches what you want.

How to read the study settings:
- `study_truth` sets the true values used to generate the synthetic data
- a scalar means fixed truth
- `sweep_axis(...)` means the study sweeps that true parameter
- every fitted parameter can have its own range, point count, and linear/log spacing
- the runtime estimate gives you a preview before the real study starts

In [ ]:
study_template_name = sample_template_name

study_templates = {
    'one_layer_drude': {
        'truth': {
            'layers[0].thickness_um': sweep_axis(350.0, 750.0, 10, scale='linear'),
            'layers[0].material.eps_inf': 11.0,
            'layers[0].material.plasma_freq_thz': sweep_axis(0.2, 2.5, 10, scale='linear'),
            'layers[0].material.gamma_thz': sweep_axis(0.01, 1.0, 4, scale='log'),
        },
        'noise_dynamic_range_db': sweep_axis(50.0, 110.0, 5, scale='linear'),
    },
    'two_layer_lorentz_plus_si': {
        'truth': {
            'layers[0].thickness_um': sweep_axis(80.0, 320.0, 10, scale='linear'),
            'layers[0].material.oscillators[0].resonance_thz': sweep_axis(0.45, 1.25, 10, scale='linear'),
            'layers[0].material.oscillators[1].resonance_thz': sweep_axis(1.2, 2.4, 10, scale='linear'),
            'layers[1].thickness_um': sweep_axis(250.0, 850.0, 2, scale='linear'),
            'layers[0].material.eps_inf': 3.4,
            'layers[0].material.oscillators[0].delta_eps': 1.2,
            'layers[0].material.oscillators[0].gamma_thz': 0.12,
            'layers[0].material.oscillators[1].delta_eps': 0.8,
            'layers[0].material.oscillators[1].gamma_thz': 0.2,
        },
        'noise_dynamic_range_db': 80.0,
    },
    'three_layer_lorentz_plus_si_plus_sio2_backside': {
        'truth': {
            'layers[0].thickness_um': sweep_axis(80.0, 320.0, 10, scale='linear'),
            'layers[0].material.oscillators[0].resonance_thz': sweep_axis(0.45, 1.25, 10, scale='linear'),
            'layers[0].material.oscillators[1].resonance_thz': sweep_axis(1.2, 2.4, 10, scale='linear'),
            'layers[1].thickness_um': sweep_axis(250.0, 850.0, 2, scale='linear'),
            'layers[0].material.eps_inf': 3.4,
            'layers[0].material.oscillators[0].delta_eps': 1.2,
            'layers[0].material.oscillators[0].gamma_thz': 0.12,
            'layers[0].material.oscillators[1].delta_eps': 0.8,
            'layers[0].material.oscillators[1].gamma_thz': 0.2,
            'layers[2].thickness_um': 1.0e7,
        },
        'noise_dynamic_range_db': 80.0,
    },
}

study_truth = deepcopy(study_templates[study_template_name]['truth'])
noise_dynamic_range_db = deepcopy(study_templates[study_template_name]['noise_dynamic_range_db'])
replicates = 1
seed = 123
seed_stride = 1000
metric = 'data_fit'
max_internal_reflections = 4
optimizer = {
    'method': 'L-BFGS-B',
    'options': {'maxiter': 120},
    'global_options': {'maxiter': 8, 'popsize': 8, 'seed': 123, 'polish': False},
    'fd_rel_step': 1e-5,
}

noise_preview_dynamic_range_db = 80.0
noise_preview_seed = 123
show_noise_fft = True
noise_fft_limits_thz = None
pilot_case_count = 10

study_definition_py_path = python_dir / 'part4_study_definition.py'
study_setup_json_path = config_dir / 'study_setup.json'
study_output_dir = run_dir / 'study'

study_config = {
    'truth': deepcopy(study_truth),
    'noise_dynamic_range_db': deepcopy(noise_dynamic_range_db),
    'replicates': replicates,
    'seed': seed,
    'seed_stride': seed_stride,
    'metric': metric,
    'max_internal_reflections': max_internal_reflections,
    'optimizer': deepcopy(optimizer),
    'out_dir': str(study_output_dir),
}

noise_preview = preview_study_noise(
    reference_result=reference_result,
    sample_result=sample_result,
    measurement=measurement,
    noise_dynamic_range_db=noise_preview_dynamic_range_db,
    max_internal_reflections=max_internal_reflections,
    seed=noise_preview_seed,
    show_fft=show_noise_fft,
    freq_limits_thz=noise_fft_limits_thz,
)
print(f'Noise sigma used in preview: {noise_preview["noise_sigma"]:.6g}')

runtime_estimate = estimate_study_runtime(
    reference_result,
    sample_result,
    study_config,
    measurement=measurement,
    pilot_case_count=pilot_case_count,
)
print_mapping('Estimated Runtime', runtime_estimate)

write_python_snapshot(
    study_definition_py_path,
    sample_template_name=sample_template_name,
    sample_definition=sample_definition,
    study_truth=study_truth,
    noise_dynamic_range_db=noise_dynamic_range_db,
    replicates=replicates,
    seed=seed,
    seed_stride=seed_stride,
    metric=metric,
    max_internal_reflections=max_internal_reflections,
    optimizer=optimizer,
    pilot_case_count=pilot_case_count,
)
save_study_setup_snapshot(
    study_setup_json_path,
    reference=reference_setup_snapshot,
    layers=layers,
    measurement=measurement,
    study=study_config,
    n_in=sample_n_in,
    n_out=sample_n_out,
    overlay_imported=overlay_imported_nk,
    sample_out_dir=run_dir / 'sample_preview',
    notes='Created from THzSim_User_Workflow.ipynb',
)
print(f'Saved machine-readable setup: {study_setup_json_path}')
print(f'Saved human-readable snapshot: {study_definition_py_path}')

## 5. Run The Study With Progress And ETA

Run the next cell only after the previews and runtime estimate look sensible.

The study saves:
- all simulated noisy traces and fitted traces per case
- the full study summary CSV
- correlation / sigma information
- heatmaps for the main quality metrics and for `true - fitted` parameter errors

In [ ]:
study_result = run_study_with_progress(
    reference_result,
    sample_result,
    study_config,
    measurement=measurement,
    out_dir=study_output_dir,
)
print(f'Study outputs: {study_result.out_dir}')
print(f'Summary CSV: {study_result.summary_csv_path}')
print(f'Progress JSON: {study_result.artifact_paths["study_progress_json"]}')
print_mapping('First Study Row', study_result.summary_rows[0])

## 6. Study Plotter And Case Browser

Edit the next cell to choose the heatmap axes and the color value.

What to do here:
- choose the `x_key` and `y_key` you want on the heatmap
- choose the `value_key` you want to color by (`data_fit`, `fit_sigma`, `normalized_mse`, `relative_l2`, or any saved `signed_err__...` field)
- choose an approximate `x` and `y` value to inspect the nearest saved case
- the notebook marks that selected case on the heatmap and then plots its traces

In [ ]:
swept_axis_keys = [key for key, value in study_truth.items() if isinstance(value, list)]
if isinstance(noise_dynamic_range_db, list) and 'noise_dynamic_range_db' not in swept_axis_keys:
    swept_axis_keys = ['noise_dynamic_range_db'] + swept_axis_keys
if len(swept_axis_keys) < 2:
    raise RuntimeError(
        'The study heatmap plotter needs at least two swept axes. '
        'In Part 4 make at least two study parameters list-valued, then rerun Part 6.'
    )
plot_x_key = 'noise_dynamic_range_db' if isinstance(noise_dynamic_range_db, list) else swept_axis_keys[0]
plot_y_key = next(key for key in swept_axis_keys if key != plot_x_key)
plot_value_key = 'data_fit'  # also try 'fit_sigma', 'normalized_mse', 'relative_l2', or any 'signed_err__...'
plot_color_scale = 'linear'  # 'linear' or 'log'
selected_x_value = study_result.summary_rows[0][plot_x_key]
selected_y_value = study_result.summary_rows[0][plot_y_key]
show_case_fft = True
case_fft_limits_thz = None

selected_row = find_nearest_study_row(
    study_result,
    x_key=plot_x_key,
    x_value=selected_x_value,
    y_key=plot_y_key,
    y_value=selected_y_value,
)
plot_study_heatmap_selector(
    study_result,
    x_key=plot_x_key,
    y_key=plot_y_key,
    value_key=plot_value_key,
    color_scale=plot_color_scale,
    selected_row=selected_row,
)
plot_study_case_detail(
    study_result,
    row=selected_row,
    show_fft=show_case_fft,
    freq_limits_thz=case_fft_limits_thz,
)
print_mapping('Selected Study Row', selected_row)